In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install -Uq wandb

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("WANDB_API_KEY")

! wandb login $secret_value_0

In [ ]:
import random

import wandb

# Start a new wandb run to track this script.
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="23f2004513-dl-genai-project",
    # Set the wandb project where this run will be logged.
    project="23f2004513-t22026",
    # Track hyperparameters and run metadata.
    config={
        "learning_rate": 0.02,
        "architecture": "RNN",
        "dataset": "CIFAR-100",
        "epochs": 10,
    },
)

# Simulate training.
epochs = 10
offset = random.random() / 5
for epoch in range(2, epochs):
    acc = 1 - 2**-epoch - random.random() / epoch - offset
    loss = 2**-epoch + random.random() / epoch + offset

    # Log metrics to wandb.
    run.log({"acc": acc, "loss": loss})

# Finish the run and upload any remaining data.
run.finish()

# 1. Important Imports

In [ ]:
import re
import random
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

#  Loading the Dataset

In [ ]:
TRAIN_PATH = "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
TEST_PATH  = "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)
train_df.head(3)

# Sanity Checks and Missing Text

In [ ]:

print("Unique answer labels:", sorted(train_df["answer"].unique()))
print("\nAnswer distribution:\n", train_df["answer"].value_counts())
print("\nMissing values:\n", train_df.isna().sum())

## 2. EDA with visualisation

In [ ]:
def word_count(text):
    return len(re.sub(r"[^a-z0-9\s]", " ", str(text).lower()).split())

# Word counts, computed once and reused for all the plots below
prompt_lengths = train_df["prompt"].apply(word_count)
option_lengths = pd.concat([train_df[c].apply(word_count) for c in OPTION_COLS], ignore_index=True)


option_rows = pd.concat([
    pd.DataFrame({
        "length": train_df[c].apply(word_count),
        "is_correct": train_df["answer"] == c,
    })
    for c in OPTION_COLS
], ignore_index=True)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

train_df["answer"].value_counts().sort_index().plot(kind="bar", ax=axes[0, 0], color="#2E5090")
axes[0, 0].set_title("Correct-answer label distribution")
axes[0, 0].set_xlabel("Option letter"); axes[0, 0].set_ylabel("Count")

axes[0, 1].hist(prompt_lengths, bins=25, color="#2E5090")
axes[0, 1].axvline(prompt_lengths.quantile(0.95), color="red", linestyle="--", label="95th pct")
axes[0, 1].set_title("Prompt length (words)")
axes[0, 1].set_xlabel("Word count"); axes[0, 1].legend()

axes[1, 0].hist(option_lengths, bins=25, color="#2E5090")
axes[1, 0].set_title("Option length, all A-E combined (words)")
axes[1, 0].set_xlabel("Word count")

option_rows.boxplot(column="length", by="is_correct", ax=axes[1, 1])
axes[1, 1].set_title("Option length: correct vs. wrong")
axes[1, 1].set_xlabel("Is the correct answer?"); axes[1, 1].set_ylabel("Word count")
plt.suptitle("")  # remove default pandas boxplot title

plt.tight_layout()
plt.show()

## 4. Preprocessing



In [ ]:
def clean_text(text):
    """Lowercase, strip punctuation/digits-noise, collapse whitespace.
    Returns "" for NaN instead of crashing -- some option cells could be empty."""
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text):
    return text.split()

### Build vocabulary from training data

In [ ]:

MIN_FREQ = 2

counter = Counter()
for _, row in train_df.iterrows():
    counter.update(tokenize(clean_text(row["prompt"])))
    for col in OPTION_COLS:
        counter.update(tokenize(clean_text(row[col])))

vocab = {"<PAD>": 0, "<UNK>": 1}
for word, freq in counter.items():
    if freq >= MIN_FREQ:
        vocab[word] = len(vocab)

VOCAB_SIZE = len(vocab)
print("Vocabulary size:", VOCAB_SIZE)

### Add text encoding function

In [ ]:
MAX_LEN = 40  # justified in the EDA plot above (comfortably past the 95th percentile)

def encode(text, max_len=MAX_LEN):
    """Text -> fixed-length list of token ids using our own vocab."""
    tokens = tokenize(clean_text(text))
    ids = [vocab.get(t, vocab["<UNK>"]) for t in tokens[:max_len]]
    ids = ids + [vocab["<PAD>"]] * (max_len - len(ids))
    return ids

## 4. Dataset Preparation


In [ ]:
# Build (question_id, prompt_ids, option_ids, label) rows
rows = []
for _, row in train_df.iterrows():
    prompt_ids = encode(row["prompt"])
    for col in OPTION_COLS:
        option_ids = encode(row[col])
        label = 1 if row["answer"] == col else 0
        rows.append((row["id"], prompt_ids, option_ids, label))

print("Total (prompt, option) training rows:", len(rows))

# Split by question id: 85% train / 15% validation
qids = train_df["id"].unique()
rng = np.random.RandomState(SEED)
rng.shuffle(qids)
split_idx = int(0.85 * len(qids))
train_qids, val_qids = set(qids[:split_idx]), set(qids[split_idx:])

train_rows = [r for r in rows if r[0] in train_qids]
val_rows   = [r for r in rows if r[0] in val_qids]
print(f"Train rows: {len(train_rows)} | Val rows: {len(val_rows)}")

In [ ]:
class MCQDataset(Dataset):
    """Each item = (prompt_ids, option_ids, label) for one (question, option) pair."""
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        _, prompt_ids, option_ids, label = self.rows[idx]
        return (
            torch.tensor(prompt_ids, dtype=torch.long),
            torch.tensor(option_ids, dtype=torch.long),
            torch.tensor(label, dtype=torch.float32),
        )


BATCH_SIZE = 64
train_loader = DataLoader(MCQDataset(train_rows), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(MCQDataset(val_rows), batch_size=BATCH_SIZE)

## 5. Model - Built & Trained From Scratch



In [ ]:
class SharedEncoder(nn.Module):
    """Embeds + BiLSTM-encodes a token sequence into one fixed-size vector
    (mean-pooled over non-pad tokens -- see reasoning above on why mean over last-hidden)."""
    def __init__(self, vocab_size, emb_dim=64, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True, bidirectional=True)

    def forward(self, token_ids):
        pad_mask = (token_ids != 0).unsqueeze(-1).float()
        embedded = self.embedding(token_ids)
        lstm_out, _ = self.lstm(embedded)
        lstm_out = lstm_out * pad_mask          # zero out padded positions before pooling
        pooled = lstm_out.sum(1) / pad_mask.sum(1).clamp(min=1)
        return pooled